## DWPOSE

In [ ]:
# !pip install controlnet-dwpose opencv-python onnxruntime-gpu huggingface_hub

In [ ]:
import os, json, time, cv2
import numpy as np
from tqdm import tqdm
from decord import VideoReader, cpu
from controlnet_dwpose import DWposeDetector

# ---------- locate project root (walks up until it finds 2d/raw_video) ----------
def project_root(start=None, marker="2d/raw_video"):
    p = os.path.abspath(start or os.getcwd())
    while True:
        if os.path.isdir(os.path.join(p, marker)):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            raise RuntimeError(f"project root not found (looking for {marker!r})")
        p = parent

ROOT       = project_root()
MODELS_DIR = r"D:\cache\huggingface"
DEVICE     = "cuda"
STYLE      = "la_hiphop"
VIDEO_NAME = "gJS_sFM_c01_d01_mJS0_ch01.mp4"

video_path = os.path.join(ROOT, "2d", "raw_video", STYLE, VIDEO_NAME)
model_det  = os.path.join(MODELS_DIR, "yolox_l.onnx")
model_pose = os.path.join(MODELS_DIR, "dw-ll_ucoco_384.onnx")

# ---------- verify everything is on disk ----------
for label, p in [("video", video_path), ("yolox", model_det), ("dwpose", model_pose)]:
    assert os.path.isfile(p), f"MISSING {label}: {p}"
    print(f"{label:8s} {os.path.getsize(p)/1e6:8.1f} MB  {p}")

import onnxruntime as ort
print("\navailable providers:", ort.get_available_providers())
assert (DEVICE == "cpu") or ("CUDAExecutionProvider" in ort.get_available_providers()), \
       "device='cuda' but CUDAExecutionProvider not available — pip install onnxruntime-gpu"

In [ ]:
# ============================================================
#  DWPose: video → per-frame keypoints → OpenPose-style JSON
# ============================================================
SAMPLE_STRIDE = 1      # 1 = every frame; 5 or 10 = faster smoke test

# ---------- 1. detector ----------
detector = DWposeDetector(model_det=model_det, model_pose=model_pose, device=DEVICE)

# ---------- 2. read video metadata ----------
vr        = VideoReader(video_path, ctx=cpu(0))
fps       = float(vr.get_avg_fps())
n_frames  = len(vr)
H, W      = vr[0].shape[:2]
indices   = list(range(0, n_frames, SAMPLE_STRIDE))
print(f"video: {VIDEO_NAME}  {n_frames} frames @ {fps:.2f} fps  {W}x{H}")
print(f"sampling {len(indices)} frames  (stride={SAMPLE_STRIDE})")

# ---------- 3. run DWPose on each sampled frame ----------
raw_poses = []
t0 = time.time()
for i in tqdm(indices, desc="DWPose"):
    frame = vr[i].asnumpy()          # HxWx3 uint8, RGB
    raw_poses.append(detector(frame))
dt = time.time() - t0
print(f"\ndone in {dt:.1f}s  ({dt/max(1, len(indices)):.3f}s/frame)")
detector.release_memory()

# ---------- 4. inspect structure of one frame ----------
p0 = raw_poses[0]
print("frame[0] keys    :", list(p0.keys()))
print("bodies.candidate :", np.asarray(p0["bodies"]["candidate"]).shape,
      " bodies.subset :", np.asarray(p0["bodies"]["subset"]).shape)
print("hands            :", len(p0.get("hands", [])), "detection(s)")
print("faces            :", len(p0.get("faces", [])), "detection(s)")
print("keypoint[0] (normalized 0..1):", np.asarray(p0["bodies"]["candidate"])[0])

# ---------- 5. helpers: flatten to OpenPose layout, de-normalize to pixels ----------
def _flatten(cand, subset, W, H):
    cand   = np.asarray(cand,   dtype=np.float32).copy()
    subset = np.asarray(subset)
    if cand.size == 0 or subset.size == 0:
        return []
    cand[:, 0] *= W
    cand[:, 1] *= H
    out = []
    for row in subset:
        ids = row[row >= 0].astype(int)
        if len(ids):
            out.append(cand[ids].flatten().tolist())
    return out

def body_people(pose, W, H):
    return [{"pose_keypoints_2d": kp}
            for kp in _flatten(pose["bodies"]["candidate"],
                               pose["bodies"]["subset"], W, H)]

def hand_people(pose, W, H):
    hands = []
    for h in pose.get("hands", []):
        for kp in _flatten(h["candidate"], h["subset"], W, H):
            hands.append({"hand_keypoints_2d": kp})
    return hands

def face_people(pose, W, H):
    faces = []
    for f in pose.get("faces", []):
        for kp in _flatten(f["candidate"], f["subset"], W, H):
            faces.append({"face_keypoints_2d": kp})
    return faces

# ---------- 6. build JSON document ----------
json_doc = {
    "video":   VIDEO_NAME,
    "style":   STYLE,
    "width":   W,
    "height":  H,
    "fps":     fps,
    "stride":  SAMPLE_STRIDE,
    "frames":  [],
}
for i, (src_idx, pose) in enumerate(zip(indices, raw_poses)):
    json_doc["frames"].append({
        "frame_index":  i,
        "source_frame": src_idx,
        "time_sec":     round(src_idx / fps, 3),
        "people":       body_people(pose, W, H),
        "hands":        hand_people(pose, W, H),
        "faces":        face_people(pose, W, H),
    })

# ---------- 7. save to 2d/keypoints/<style>/<stem>.json ----------
stem    = os.path.splitext(VIDEO_NAME)[0]
out_dir = os.path.join(ROOT, "2d", "keypoints", STYLE)
os.makedirs(out_dir, exist_ok=True)
OUT_JSON = os.path.join(out_dir, stem + ".json")

with open(OUT_JSON, "w") as f:
    json.dump(json_doc, f, indent=2)
print(f"\nsaved: {OUT_JSON}")
print(f"       {os.path.getsize(OUT_JSON)/1024:.1f} KB, {len(json_doc['frames'])} frames")

# ---------- 8. visual check on frame 0 ----------
import matplotlib.pyplot as plt
vis  = np.array(vr[0].asnumpy())
cand = np.asarray(raw_poses[0]["bodies"]["candidate"], dtype=np.float32).copy()
cand[:, 0] *= W; cand[:, 1] *= H
for x, y, c in cand:
    if c > 0.3:
        cv2.circle(vis, (int(x), int(y)), 5, (0, 255, 0), -1)
plt.figure(figsize=(10, 6)); plt.imshow(vis); plt.axis("off")
plt.title(f"{VIDEO_NAME} — frame 0, {len(cand)} keypoints")
plt.show()